# Assignment 1.2: RAG-Based Intelligent Document Assistant
### (LangChain + LlamaIndex)

**Objective:** Build an AI-powered Document Question Answering system using **RAG (Retrieval-Augmented Generation)** that:

- Ingests documents (PDF, TXT, MD, DOCX and web pages)
- Indexes them using **LlamaIndex**
- Performs semantic retrieval using **embeddings**
- Uses **LangChain** for LLM orchestration and response generation
- Answers user queries with context-aware, accurate, source-cited responses
- Supports **multi-document querying**, **document comparison** and **summarization**



### Step 1: Import libraries and load environment variables

In [2]:
import os
import json
import hashlib
import warnings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from typing import List, Literal, Optional

import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel, Field
from dotenv import load_dotenv

# Supressing warnings for the sake of a clean output
warnings.filterwarnings("ignore")

# Load environment variables from .env file
load_dotenv()

# Which LLM provider to use: "groq" (default) or "openai"
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "groq").lower()


def missing_api_key() -> Optional[str]:
    """Return the name of the missing API key, or None if the key is present."""
    key_name = "OPENAI_API_KEY" if LLM_PROVIDER == "openai" else "GROQ_API_KEY"
    return None if os.getenv(key_name) else key_name

d:\Vs Code\Langchain_Langraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
if missing_api_key():
    print(f"WARNING: {missing_api_key()} not found in .env file")
else:
    print("Environment variables loaded successfully. LLM provider:", LLM_PROVIDER)

Environment variables loaded successfully. LLM provider: groq


### Step 2: Configuration

All the important knobs of the RAG system are kept in one place.

- **CHUNK_SIZE / CHUNK_OVERLAP** – how big each piece of a document is (in tokens) and how much neighbouring chunks overlap.
- **TOP_K** – how many chunks the retriever brings back for each question.
- **EMBED_MODEL_NAME** – a free, local HuggingFace embedding model (no API key needed, 384 dimensions).
- **Web URLs** are saved in `storage/web_urls.json`, so URLs added here or in the web app are used by both.

In [4]:
# Folder paths
DATA_DIR = "data"          # put your PDFs / TXT / MD / DOCX files here
STORAGE_DIR = "storage"    # LlamaIndex will save the vector index here
URLS_FILE = os.path.join(STORAGE_DIR, "web_urls.json")
FINGERPRINT_FILE = os.path.join(STORAGE_DIR, "fingerprint.json")

# Supported file types
SUPPORTED_EXTS = [".pdf", ".txt", ".md", ".docx"]

# Chunking + retrieval settings
CHUNK_SIZE = 512
CHUNK_OVERLAP = 50
TOP_K = 4
MAX_CHARS_PER_CALL = 6000    # summarization: text sent per LLM call (keeps each request small)

# Groq free tier allows only a few thousand tokens per minute (TPM).
# These settings keep requests small and wait + retry when the limit is hit.
MAX_OUTPUT_TOKENS = 2048      # cap on the length of each LLM answer
RATE_LIMIT_RETRIES = 4        # how many times to retry after a rate-limit error
RATE_LIMIT_WAIT_SECONDS = 30  # wait time before retrying

# Models
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
INDEX_VERSION = 2   # change this number to force every saved index to be rebuilt
GROQ_MODEL = "openai/gpt-oss-20b"
OPENAI_MODEL = "gpt-4o-mini"
TEMPERATURE = 0.2

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(STORAGE_DIR, exist_ok=True)


# Web pages to ingest (stored in a small JSON file)
def load_saved_urls() -> List[str]:
    if os.path.exists(URLS_FILE):
        with open(URLS_FILE) as f:
            return json.load(f)
    return []


def save_urls(urls: List[str]):
    with open(URLS_FILE, "w") as f:
        json.dump(urls, f)


def add_url(url: str):
    urls = load_saved_urls()
    if url.strip() and url.strip() not in urls:
        save_urls(urls + [url.strip()])

In [5]:
# Optional: add a web page to the knowledge base (uncomment to use)
# add_url("https://en.wikipedia.org/wiki/Retrieval-augmented_generation")

print("Configuration ready.")
print("Web URLs:", load_saved_urls())

Configuration ready.
Web URLs: []


### Step 3: LLM integration using LangChain

LangChain gives one common interface (`.invoke()`) for every LLM provider. We use **ChatGroq** by default and **ChatOpenAI** as an alternative.

In [6]:
def get_llm():
    """Return a LangChain chat model based on LLM_PROVIDER."""
    if LLM_PROVIDER == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE, max_tokens=MAX_OUTPUT_TOKENS)

    from langchain_groq import ChatGroq
    return ChatGroq(model=GROQ_MODEL, temperature=TEMPERATURE, max_tokens=MAX_OUTPUT_TOKENS)


def is_rate_limit_error(error: Exception) -> bool:
    text = str(error).lower()
    return "rate_limit" in text or "rate limit" in text or "429" in text or "413" in text


def invoke_with_retry(chain, inputs: dict):
    """Invoke a LangChain chain; if the provider's rate limit is hit, wait and try again."""
    import time
    for attempt in range(RATE_LIMIT_RETRIES + 1):
        try:
            return chain.invoke(inputs)
        except Exception as e:
            if not is_rate_limit_error(e) or attempt == RATE_LIMIT_RETRIES:
                raise
            print(f"Rate limit reached - waiting {RATE_LIMIT_WAIT_SECONDS}s before retry "
                  f"({attempt + 1}/{RATE_LIMIT_RETRIES})...")
            time.sleep(RATE_LIMIT_WAIT_SECONDS)

In [7]:
llm = get_llm()

# Test the LLM to ensure it's working
print(llm.invoke("In one line, what is Retrieval-Augmented Generation?").content)

Retrieval‑Augmented Generation is a method that boosts a language model’s output by first fetching relevant external documents and then conditioning the generation on that retrieved knowledge.


### Step 4: Configure LlamaIndex (embeddings + chunking)

- `Settings.embed_model` – the embedding model LlamaIndex uses to convert text into vectors.
- `Settings.node_parser` – the `SentenceSplitter` that breaks documents into chunks (called **nodes** in LlamaIndex).
- `Settings.llm = None` – LlamaIndex is used **only** for indexing and retrieval. All answer generation is done by LangChain.

> The first run downloads the embedding model (~90 MB). After that it works offline.

In [8]:
from llama_index.core import Settings
from llama_index.core.node_parser import SentenceSplitter


def configure_llamaindex():
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding

    # Embedding model (runs locally, free)
    Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)

    # Chunking strategy
    Settings.node_parser = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

    # LLM is handled by LangChain, so we switch off LlamaIndex's own LLM
    Settings.llm = None

In [9]:
configure_llamaindex()

# Test the embedding model
test_vector = Settings.embed_model.get_text_embedding("Delhi is the capital of India")
print("Embedding dimension:", len(test_vector))
print("First 5 values:", test_vector[:5])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15853.12it/s]


LLM is explicitly disabled. Using MockLLM.
Embedding dimension: 384
First 5 values: [0.04354960098862648, 0.02387724071741104, -0.04524128884077072, 0.03540494665503502, -0.01665102131664753]


### Step 5: Document ingestion (local files + web pages)

- **Local files:** LlamaIndex `SimpleDirectoryReader` reads every supported file in `data/`. PDFs are read page by page, so each page keeps its `page_label` (used later for citations).
  > **Important:** the package `llama-index-readers-file` must be installed. Without it, LlamaIndex silently reads a PDF as raw bytes (`%PDF-1.5 ... endobj ... /FontDescriptor`) and the assistant cannot answer. The code below checks this and stops with a clear message.
- **Web pages:** fetched with `requests`, cleaned with `BeautifulSoup`, and converted into LlamaIndex `Document` objects.

In [10]:
from llama_index.core import SimpleDirectoryReader, Document


def load_local_documents(data_dir: str = DATA_DIR) -> List[Document]:
    """Load all supported files from the data folder using LlamaIndex."""
    has_files = any(
        os.path.splitext(f)[1].lower() in SUPPORTED_EXTS
        for _, _, files in os.walk(data_dir) for f in files
    )
    if not has_files:
        print(f"No supported files found in '{data_dir}'.")
        return []

    # Without this package PDFs/DOCX are read as raw bytes instead of text
    try:
        import llama_index.readers.file  # noqa: F401
    except ImportError:
        raise ImportError("PDF/DOCX reader missing. Run:  pip install llama-index-readers-file pypdf docx2txt")

    reader = SimpleDirectoryReader(
        input_dir=data_dir,
        recursive=True,
        required_exts=SUPPORTED_EXTS,
    )
    return reader.load_data()


def load_web_documents(urls: List[str]) -> List[Document]:
    """Download web pages and convert them into LlamaIndex Documents."""
    web_docs = []
    for url in urls:
        try:
            response = requests.get(url, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            # remove non-content tags
            for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
                tag.decompose()

            text = " ".join(soup.get_text(separator=" ").split())
            if text:
                web_docs.append(Document(text=text, metadata={"file_name": url, "source_type": "web"}))
                print(f"Loaded web page: {url}")
        except Exception as e:
            print(f"Could not load {url}: {e}")
    return web_docs


def load_all_documents() -> List[Document]:
    return load_local_documents(DATA_DIR) + load_web_documents(load_saved_urls())

In [11]:
documents = load_all_documents()

print("Total documents loaded:", len(documents))
for doc in documents:
    print("-", doc.metadata.get("file_name"), "| page:", doc.metadata.get("page_label", "-"),
          "| characters:", len(doc.text))

Total documents loaded: 2
- Akshay_Kale_Resume_v3 (3) (1).pdf | page: 1 | characters: 3432
- Akshay_Kale_Resume_v3 (3) (1).pdf | page: 2 | characters: 1994


In [12]:
# Look inside one document
if documents:
    print(documents[0].metadata)
    print("-" * 80)
    print(documents[0].text[:500])

{'page_label': '1', 'file_name': 'Akshay_Kale_Resume_v3 (3) (1).pdf', 'file_path': 'd:\\Vs Code\\Langchain_Langraph\\Stock_Assistant\\data\\Akshay_Kale_Resume_v3 (3) (1).pdf', 'file_type': 'application/pdf', 'file_size': 282702, 'creation_date': '2026-10-15', 'last_modified_date': '2026-10-15'}
--------------------------------------------------------------------------------
 
Akshay Kale 
Senior Accounts Payable Analyst 
+91 8446710587   |   akshaykale3@gmail.com   |   Pune, India 
linkedin.com/in/akshaykale-156812106 
 
ABOUT ME 
Accounts Payable professional with 9+ years of experience at Metro Global Solution Center, specializing in 
invoice processing, vendor reconciliation, and AP controls. Skilled in SAP, SQL, and automation projects, with a 
strong record of process improvements, system testing, and team training. Recognized with multiple client and 
interna


### Step 6: Build (or load) the vector index with LlamaIndex

`VectorStoreIndex.from_documents()` does three things automatically:
1. Splits each document into chunks (nodes) using our `SentenceSplitter`
2. Creates an embedding for every chunk
3. Stores the vectors in a vector store

The index is **saved to `storage/`**, so next time it loads instantly instead of re-embedding everything.
A small *fingerprint* (file names, sizes, dates, URLs and settings) tells us when the documents changed and the index must be rebuilt.

In [13]:
from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage


def get_sources_fingerprint() -> str:
    """A hash that changes whenever files, URLs or index settings change."""
    items = []
    for root, _, files in os.walk(DATA_DIR):
        for f in files:
            if os.path.splitext(f)[1].lower() in SUPPORTED_EXTS:
                path = os.path.join(root, f)
                stat = os.stat(path)
                items.append(f"{path}|{stat.st_size}|{int(stat.st_mtime)}")
    items += [f"url|{u}" for u in load_saved_urls()]
    items.append(f"settings|{CHUNK_SIZE}|{CHUNK_OVERLAP}|{EMBED_MODEL_NAME}|v{INDEX_VERSION}")
    return hashlib.md5("\n".join(sorted(items)).encode()).hexdigest()


def build_or_load_index(force_rebuild: bool = False) -> VectorStoreIndex:
    """Load the saved index if documents did not change, otherwise rebuild it."""
    fingerprint = get_sources_fingerprint()

    saved_fingerprint = None
    if os.path.exists(FINGERPRINT_FILE):
        with open(FINGERPRINT_FILE) as f:
            saved_fingerprint = json.load(f).get("fingerprint")

    if not force_rebuild and saved_fingerprint == fingerprint:
        print("Loading existing index from storage...")
        storage_context = StorageContext.from_defaults(persist_dir=STORAGE_DIR)
        return load_index_from_storage(storage_context)

    print("Building a new index...")
    docs = load_all_documents()
    if not docs:
        raise ValueError(f"No documents to index. Add files to '{DATA_DIR}' or add a web URL.")

    index = VectorStoreIndex.from_documents(docs, show_progress=True)
    index.storage_context.persist(persist_dir=STORAGE_DIR)

    with open(FINGERPRINT_FILE, "w") as f:
        json.dump({"fingerprint": fingerprint}, f)

    return index


def list_indexed_documents(index: VectorStoreIndex) -> List[str]:
    """Names of all documents present in the index (used for multi-document features)."""
    return sorted({node.metadata.get("file_name", "unknown") for node in index.docstore.docs.values()})

In [14]:
index = build_or_load_index()

print("Number of chunks (nodes) in the index:", len(index.docstore.docs))

available_docs = list_indexed_documents(index)
print("Documents available in the index:")
for name in available_docs:
    print("-", name)

2026-10-15 22:59:01,997 - INFO - Loading all indices.


Loading existing index from storage...
Number of chunks (nodes) in the index: 3
Documents available in the index:
- Akshay_Kale_Resume_v3 (3) (1).pdf


### Step 7: Semantic retrieval with the LlamaIndex Retriever

The retriever converts the question into an embedding and returns the **top-k most similar chunks** (cosine similarity).
We can also pass **metadata filters** to search inside specific documents only – this is what powers multi-document querying.

In [15]:
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator, FilterCondition


def get_retriever(index: VectorStoreIndex, top_k: int = TOP_K, file_names: Optional[List[str]] = None):
    """LlamaIndex retriever. If file_names is given, search only inside those documents."""
    filters = None
    if file_names:
        # One "file_name == X" filter per document, combined with OR
        # (simple string comparisons work the same in every LlamaIndex version)
        filters = MetadataFilters(
            filters=[MetadataFilter(key="file_name", value=str(name), operator=FilterOperator.EQ)
                     for name in file_names],
            condition=FilterCondition.OR,
        )
    return index.as_retriever(similarity_top_k=top_k, filters=filters)

In [16]:
retriever = get_retriever(index)

# Test the retriever
test_nodes = retriever.retrieve("What is the price tolerance for three-way match?")

for node in test_nodes:
    print(f"Score: {node.score:.3f} | Source: {node.metadata.get('file_name')}")
    print(node.text[:200].replace("\n", " "), "...")
    print("-" * 80)

Score: 0.081 | Source: Akshay_Kale_Resume_v3 (3) (1).pdf
• Supported the smooth transfer of new AP processes from client teams into GSC, checking data quality and  making sure nothing was missed during handover.  Analyst – Accounts Payable Jul 2018 – Sep 20 ...
--------------------------------------------------------------------------------
Score: 0.009 | Source: Akshay_Kale_Resume_v3 (3) (1).pdf
• Worked closely with the transition team to bring in new AP activities from country colleagues — gathered  process knowledge, documented the steps, and got the work running smoothly within GSC.   • P ...
--------------------------------------------------------------------------------
Score: -0.015 | Source: Akshay_Kale_Resume_v3 (3) (1).pdf
Akshay Kale  Senior Accounts Payable Analyst  +91 8446710587   |   akshaykale3@gmail.com   |   Pune, India  linkedin.com/in/akshaykale-156812106    ABOUT ME  Accounts Payable professional with 9+ year ...
---------------------------------------------------

### Step 8: Connect LlamaIndex to LangChain

The retrieved nodes are formatted into one **context** string with source labels like `[Source 1: file.pdf, page 3]`.
Then we wrap the LlamaIndex retriever inside a LangChain **`RunnableLambda`**, so it can be used directly inside an LCEL chain with the `|` operator.

In [17]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough


def source_label(node) -> str:
    name = node.metadata.get("file_name", "unknown")
    page = node.metadata.get("page_label")
    return f"{name}, page {page}" if page else name


def format_nodes(nodes) -> str:
    """Convert retrieved LlamaIndex nodes into a context string for the prompt."""
    if not nodes:
        return "No relevant context found."
    blocks = []
    for i, node in enumerate(nodes, start=1):
        blocks.append(f"[Source {i}: {source_label(node)}]\n{node.get_content().strip()}")
    return "\n\n".join(blocks)


def make_context_runnable(retriever):
    """LlamaIndex retriever  ->  LangChain Runnable that returns the context string."""
    return RunnableLambda(lambda question: format_nodes(retriever.retrieve(question)))

In [18]:
context_runnable = make_context_runnable(retriever)
print(context_runnable.invoke("When are payment runs executed?")[:600])

[Source 1: Akshay_Kale_Resume_v3 (3) (1).pdf, page 2]
• Supported the smooth transfer of new AP processes from client teams into GSC, checking data quality and 
making sure nothing was missed during handover. 
Analyst – Accounts Payable Jul 2018 – Sep 2020 
Metro Global Solution Center (Metro-GSC), Pune 
• Handled end-to-end invoice processing in SAP — keying in data, routing for approvals, and ensuring 
invoices were correctly posted — consistently achieving high accuracy scores. 
• Resolved payment-related disputes by investigating invoice discrepancies, coordinating with vendors, and 
worki


### Step 9: Prompt templates (LangChain)

We create re-usable **ChatPromptTemplates** with a system message (rules) and a user message (context + question).
The rules force the LLM to stay grounded in the documents and to cite sources – this reduces hallucinations.

In [19]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are an intelligent Document Assistant.
Answer the user's question using ONLY the context provided below.

Rules:
1. Use only the information in the context. Never fabricate information.
2. If the context does not contain the answer, say exactly: "I could not find this information in the provided documents."
3. Cite the sources you used in square brackets, for example [Source 1] or [Source 2].
4. Keep the answer clear and well structured. Use bullet points where helpful."""),
    ("user",
     """Context:
{context}

Question: {question}

Answer:""")
])

In [20]:
print(qa_prompt.invoke({"context": "<retrieved chunks>", "question": "<user question>"}))

messages=[SystemMessage(content='You are an intelligent Document Assistant.\nAnswer the user\'s question using ONLY the context provided below.\n\nRules:\n1. Use only the information in the context. Never fabricate information.\n2. If the context does not contain the answer, say exactly: "I could not find this information in the provided documents."\n3. Cite the sources you used in square brackets, for example [Source 1] or [Source 2].\n4. Keep the answer clear and well structured. Use bullet points where helpful.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Context:\n<retrieved chunks>\n\nQuestion: <user question>\n\nAnswer:', additional_kwargs={}, response_metadata={})]


### Step 10: RAG chain using LCEL (LangChain Expression Language)

The `|` operator pipes the output of one step into the next:

```
question ──► {context: LlamaIndex retriever, question: passthrough} ──► prompt ──► LLM ──► StrOutputParser ──► answer
```

In [21]:
from langchain_core.output_parsers import StrOutputParser


def build_rag_chain(retriever, llm):
    return (
        {
            "context": make_context_runnable(retriever),   # LlamaIndex retrieval
            "question": RunnablePassthrough(),             # user question as-is
        }
        | qa_prompt
        | llm
        | StrOutputParser()
    )

In [22]:
rag_chain = build_rag_chain(retriever, llm)

question = "What details must a valid supplier invoice contain?"
print("Question:", question)
print("-" * 80)
print(rag_chain.invoke(question))

Question: What details must a valid supplier invoice contain?
--------------------------------------------------------------------------------


2026-10-15 22:59:02,615 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m268rmthefrbvpwdyb25tcs3` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198463, Requested 1706. Please try again in 1m13.008s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [23]:
# A question that is NOT in the documents - the assistant should refuse instead of guessing
print(rag_chain.invoke("Who won the IPL 2026?"))

2026-10-15 22:59:25,059 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m268rmthefrbvpwdyb25tcs3` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198411, Requested 1760. Please try again in 1m13.872s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

### Step 11: Structured output with PydanticOutputParser

For applications, a structured answer is more useful than plain text. We define a **Pydantic model** and let LangChain's
`PydanticOutputParser` add format instructions to the prompt and convert the LLM output into a Python object.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser


class RAGAnswer(BaseModel):
    answer: str = Field(description="The answer to the question, based only on the context")
    sources: List[str] = Field(description="Document names used for the answer, e.g. ['policy.pdf, page 2']")
    confidence: Literal["high", "medium", "low"] = Field(description="How well the context supports the answer")
    follow_up_questions: List[str] = Field(description="Two short follow-up questions the user might ask next")


pydantic_parser = PydanticOutputParser(pydantic_object=RAGAnswer)

structured_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are an intelligent Document Assistant. Answer using ONLY the context.
If the context does not contain the answer, say so in the answer field and set confidence to "low".

{format_instructions}"""),
    ("user", "Context:\n{context}\n\nQuestion: {question}")
]).partial(format_instructions=pydantic_parser.get_format_instructions())


def generate_structured_answer(context: str, question: str, llm, fallback_sources: List[str]) -> RAGAnswer:
    """Structured answer. Falls back to plain text if the LLM output can't be parsed."""
    inputs = {"context": context, "question": question}
    try:
        return invoke_with_retry(structured_prompt | llm | pydantic_parser, inputs)
    except Exception as e:
        if is_rate_limit_error(e):
            raise
        text = invoke_with_retry(qa_prompt | llm | StrOutputParser(), inputs)
        return RAGAnswer(answer=text, sources=fallback_sources, confidence="medium", follow_up_questions=[])


def ask_structured(question: str, retriever, llm) -> RAGAnswer:
    nodes = retriever.retrieve(question)
    return generate_structured_answer(format_nodes(nodes), question, llm,
                                      sorted({source_label(n) for n in nodes}))

In [ ]:
result = ask_structured("What is the approval limit for non-PO invoices?", retriever, llm)

print("Answer:", result.answer)
print("Sources:", result.sources)
print("Confidence:", result.confidence)
print("Follow-up questions:", result.follow_up_questions)

### Step 12: Multi-document querying

**12 (a) – Query a specific document:** the retriever's metadata filter limits the search to the chosen file(s).

**12 (b) – Compare across documents:** we retrieve the best chunks **separately from each document**, label the context by document, and ask the LLM to compare.

In [ ]:
# 12 (a) Ask a question inside one selected document only
selected_doc = available_docs[-1]
doc_retriever = get_retriever(index, file_names=[selected_doc])
doc_chain = build_rag_chain(doc_retriever, llm)

print("Searching only in:", selected_doc)
print("-" * 80)
print(doc_chain.invoke("What approvals are required and what are the limits?"))

In [ ]:
compare_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a Document Analyst. You receive context grouped by document.
Answer the question for EACH document separately and then give a short comparison
(similarities and differences). Use ONLY the given context. If a document does not cover
the topic, say "Not covered in this document". Use a markdown table where helpful."""),
    ("user", "{context}\n\nQuestion: {question}")
])


def compare_documents(index, question: str, file_names: List[str], llm, top_k: int = 3) -> str:
    sections = []
    for name in file_names:
        nodes = get_retriever(index, top_k=top_k, file_names=[name]).retrieve(question)
        sections.append(f"=== DOCUMENT: {name} ===\n{format_nodes(nodes)}")
    compare_chain = compare_prompt | llm | StrOutputParser()
    return invoke_with_retry(compare_chain, {"context": "\n\n".join(sections), "question": question})

In [ ]:
# 12 (b) Compare documents
print(compare_documents(index, "What are the approval rules and amount limits?", available_docs, llm))

### Step 13: Document summarization (Map-Reduce with LangChain)

1. Collect all chunks of the selected document from the LlamaIndex docstore (in original order).
2. **Map:** if the document is long, split it into small groups and summarize each group **one at a time**
   (one call at a time keeps us under the Groq free-tier tokens-per-minute limit; `invoke_with_retry` waits and retries if the limit is hit).
3. **Reduce:** combine the partial summaries into one final summary (repeated if the partial summaries are still too long).

In [ ]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert at summarizing documents accurately. Use only the given text."),
    ("user",
     """Summarize the following text from the document "{doc_name}".

Give:
1. A one-paragraph overview
2. Key points as bullet points (include important numbers, limits and deadlines)

Text:
{text}""")
])

combine_prompt = ChatPromptTemplate.from_messages([
    ("system", "You combine partial summaries into one clear final summary without repeating points."),
    ("user",
     """These are partial summaries of the document "{doc_name}":

{text}

Write the final summary with:
1. A one-paragraph overview
2. Key points as bullet points""")
])


def get_document_text_chunks(index: VectorStoreIndex, file_name: str) -> List[str]:
    return [node.get_content() for node in index.docstore.docs.values()
            if node.metadata.get("file_name") == file_name]


def group_texts(texts: List[str], max_chars: int = MAX_CHARS_PER_CALL) -> List[str]:
    """Join texts into groups of at most max_chars characters (long single texts are split)."""
    groups, current = [], ""
    for text in texts:
        while len(text) > max_chars:                 # split a very long text
            if current:
                groups.append(current)
                current = ""
            groups.append(text[:max_chars])
            text = text[max_chars:]
        if current and len(current) + len(text) > max_chars:
            groups.append(current)
            current = ""
        current += text + "\n\n"
    if current.strip():
        groups.append(current)
    return groups


def summarize_document(index, file_name: str, llm, progress_callback=None) -> str:
    chunks = get_document_text_chunks(index, file_name)
    if not chunks:
        return f"Document '{file_name}' not found in the index."

    summary_chain = summary_prompt | llm | StrOutputParser()
    combine_chain = combine_prompt | llm | StrOutputParser()

    groups = group_texts(chunks)

    # Short document -> single call
    if len(groups) == 1:
        return invoke_with_retry(summary_chain, {"doc_name": file_name, "text": groups[0]})

    # Long document -> MAP: summarize each group one by one (stays under the rate limit)
    partial_summaries = []
    for i, group in enumerate(groups, start=1):
        message = f"Summarizing part {i} of {len(groups)}..."
        print(message)
        if progress_callback:
            progress_callback(i / len(groups), message)
        partial_summaries.append(invoke_with_retry(summary_chain, {"doc_name": file_name, "text": group}))

    # REDUCE: combine partial summaries (repeat while they are still too long for one call)
    while len(partial_summaries) > 1:
        combined_groups = group_texts(partial_summaries)
        if len(combined_groups) == 1:
            break
        partial_summaries = [invoke_with_retry(combine_chain, {"doc_name": file_name, "text": g})
                             for g in combined_groups]

    return invoke_with_retry(combine_chain, {"doc_name": file_name,
                                             "text": "\n\n---\n\n".join(partial_summaries)})

In [ ]:
print(summarize_document(index, available_docs[0], llm))

### Step 14: Conversational Document Assistant (with chat history)

Follow-up questions such as *"and on which days does that happen?"* do not make sense on their own.
So before retrieval, a small LangChain chain **rewrites the follow-up into a standalone question** using the chat history.
Then the normal RAG flow runs and the answer is returned together with its sources.

Note: normal chat only sees the **top-k chunks**, which is not enough to summarize a whole document. So `is_summary_request()`
detects questions like *"give me a summary of the pdf"* and the app sends them to `summarize_document()` (Step 13) instead.
This `ask()` function is exactly what the web app calls for every chat message.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

condense_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """Given the chat history and a follow-up question, rewrite the follow-up question
into a standalone question that can be understood without the chat history.
Do NOT answer the question. Return ONLY the rewritten question."""),
    MessagesPlaceholder("chat_history"),
    ("user", "{question}")
])


SUMMARY_WORDS = ["summary", "summarize", "summarise", "overview", "summarization"]


def is_summary_request(question: str) -> bool:
    """Chat retrieval only sees the top-k chunks, so 'summarize the document' needs summarize_document()."""
    return any(word in question.lower() for word in SUMMARY_WORDS)


def ask(question: str, chat_history: list, retriever, llm, structured: bool = False) -> dict:
    """Conversational RAG: condense question -> retrieve (LlamaIndex) -> answer (LangChain)."""
    standalone_question = question
    if chat_history:
        condense_chain = condense_prompt | llm | StrOutputParser()
        standalone_question = invoke_with_retry(condense_chain, {"chat_history": chat_history, "question": question})

    nodes = retriever.retrieve(standalone_question)
    context = format_nodes(nodes)
    sources = [{"source": source_label(n), "score": round(n.score or 0, 3),
                "text": n.get_content()[:400]} for n in nodes]

    result = {"question": question, "standalone_question": standalone_question,
              "sources": sources, "confidence": None, "follow_up_questions": []}

    if structured:
        parsed = generate_structured_answer(context, standalone_question, llm,
                                            sorted({s["source"] for s in sources}))
        result.update(answer=parsed.answer, confidence=parsed.confidence,
                      follow_up_questions=parsed.follow_up_questions)
    else:
        answer_chain = qa_prompt | llm | StrOutputParser()
        result["answer"] = invoke_with_retry(answer_chain, {"context": context, "question": standalone_question})

    return result

In [ ]:
# Demo of a follow-up conversation
chat_history = []

for q in ["What is the standard payment term for suppliers?",
          "And on which days does that payment happen?"]:
    response = ask(q, chat_history, retriever, llm)
    print("User:", q)
    print("Standalone question:", response["standalone_question"])
    print("Assistant:", response["answer"])
    print("Sources:", sorted({s["source"] for s in response["sources"]}))
    print("-" * 80)
    chat_history += [HumanMessage(content=q), AIMessage(content=response["answer"])]

### Step 15: Interactive chat loop (inside the notebook)

Type your questions. Special commands:
- `summary <file name>` – summarize a document
- `docs` – list indexed documents
- `bye` or `exit` – stop

> In VS Code the input box appears at the **top** of the window.

In [ ]:
history = []
print("Document Assistant ready! Type 'bye' or 'exit' to stop.\n")

while True:
    user_input = input("Enter query: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ["bye", "exit"]:
        print("Assistant: Goodbye!")
        break

    print(f"User: {user_input}")

    if user_input.lower() == "docs":
        print("Assistant:", ", ".join(available_docs))
    elif user_input.lower().startswith("summary "):
        print("Assistant:", summarize_document(index, user_input[8:].strip(), llm))
    elif is_summary_request(user_input):
        print("Assistant: To summarize a whole document type:  summary <file name>   (use 'docs' to see file names)")
    else:
        response = ask(user_input, history, retriever, llm)
        print("Assistant:", response["answer"])
        print("Sources:", ", ".join(sorted({s["source"] for s in response["sources"]})))
        history += [HumanMessage(content=user_input), AIMessage(content=response["answer"])]
    print("-" * 80)

### Step 16: Deployable application (Streamlit)

`rag_document_assistant.py` contains **all the code above** (the same functions) **plus a Streamlit UI**:
upload documents, add URLs, chat with sources, search selected documents, compare documents and summarize.

Run it from the VS Code terminal:

```bash
streamlit run rag_document_assistant.py
```

Or start it from this notebook with the next cell (it opens at http://localhost:8501). Run the cell after that to stop it.

In [ ]:
import subprocess
import sys

app_process = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "rag_document_assistant.py"])
print("Streamlit app started at http://localhost:8501")

In [ ]:
# Stop the Streamlit app
app_process.terminate()
print("Streamlit app stopped.")

### Conclusion

- **LlamaIndex** handled ingestion, chunking, embeddings, the persisted vector index and filtered retrieval.
- **LangChain** handled the LLM, prompt templates, LCEL chains/runnables and output parsing (String + Pydantic).
- The assistant answers only from the documents, cites sources, refuses out-of-scope questions, supports follow-up questions,
  multi-document querying/comparison and summarization, and is deployed as a Streamlit web app.